In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import os
from statsmodels.stats.weightstats import DescrStatsW

In [ ]:
location_type_info = [
    "on a helix facing the membrane core",
    "on a helix facing water or headgroups",
    "on a helix N terminal with no helix carbonyl to hydrogen bond with (variable solvation)",
    "on a loop (all of these are in water)"
]

#more concise versions of the above
site_names = ["intrahelical membrane/interior", "intrahelical aqueous", "helix N terminus (variable solvation)", "loop (aqueous)"]

#copy command:
#in: /home/jonathan/Documents/grabelab/rutter-lab-collaboration/thr_rotamers_solvation
#scp -r jborowsky@x01.ucsf.edu:/media/X01Raid01/Data_Backup/home/jborowsky/aac1-ucp1-aa-and-threonine/long-aac1-ucp1-processing/thr_rotamers/ .

In [ ]:
#this path actually also contains serine rotamers
inputpath = "/home/jonathan/Documents/grabelab/rutter-lab-collaboration/thr_rotamers_solvation/thr_solvation"

servers = ["wynton", "degrabo"]
proteins = ["aac1", "ucp1"]

resn = "thr" #"ser" #

#loop over servers, proteins, and parallel runs
for protein in proteins:

    #--------------------------------------------------------------------------
    #                  load rotamer data for each run

    rotamer_freqs_all = [] #frequency of each rotamer for each residue
    n_frames_all = [] #number of frames in run

    for server in servers:    
        for run in range(1,5):

            rotamer_data = []

            #not all servers have the same number of parallel simulations
            if run > 2 and server == "degrabo": continue

            for seg in range(1,20):

                #load data if it exists
                datafile = f"{inputpath}/{server}/{protein}/run0{run}/{protein}-{server}-run0{run}-seg{str(seg).zfill(2)}-{resn}-rotamer-counts.npy"
                if not os.path.exists(datafile): break
                data = np.load(datafile)
                
                rotamer_data.append(data)


            #--------------------------------------------------------------------
            #solvation
            nsolv = np.concatenate(nsolv)
            print(nsolv.shape)

            solv_nums_mean_byresi = []

            for i in range(len(threonines_bytype)):
                thr_nsolv = nsolv[:, thr_o_indices[i]] #the residues are not in order; see selection code in jupyter notebook.
                solv_nums_mean_byresi.append(np.mean(thr_nsolv, axis = 0))
                #std_solv = np.std(thr_nsolv, axis = 0)

            solv_nums_mean = [np.mean(sni) for sni in solv_nums_mean_byresi]
            solv_nums_std = [np.std(sni) for sni in solv_nums_mean_byresi]

            bar_inds = [i for i in range(len(threonines_bytype))]

            plt.bar(bar_inds, solv_nums_mean, yerr=solv_nums_std)

            plt.ylim(0,2.5)

            plt.xlabel("Threonine location")
            plt.ylabel("Average number of water molecules\nwithin 3.3 Å of OH group")

            tstr = "{:.2f}".format(step * 10 * nsolv.shape[0])
            plt.title(f"Threonine solvation in run {ci};\n{tstr} microseconds, {nsolv.shape[0]} frames")
            #print(f"{step * 10 * nsolv.shape[0]} microseconds")
            plt.xticks(bar_inds, ["intrahelical;\nburied", "intrahelical;\naqueous", "loop;\naqueous"]) #"helix N-terminus\n(variable solvation)", 
            plt.savefig(f"{inputpath_solv}/threonine_solvation_run{ci}.png", dpi=600, bbox_inches='tight')
            plt.show()